In [ ]:
# ▶ STEP 1 — Run this cell first to install dependencies
!pip install pybaseball pandas

In [ ]:
# ▶ STEP 2 — Fill in your lineup settings here

YEAR       = "2024"       # Season year
HANDEDNESS = "R"          # "R" for RHP, "L" for LHP

LINEUP = [
    "Mookie Betts",
    "Freddie Freeman",
    "Will Smith",
    "Teoscar Hernandez",
    "Max Muncy",
    "Tommy Edman",
    "Kiké Hernandez",
    "Andy Pages",
    "Miguel Rojas",
]

In [ ]:
# ▶ STEP 3 — Run this cell to fetch data and see results

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pybaseball import statcast_batter, playerid_lookup, cache

cache.enable()

# ── Helpers ────────────────────────────────────────────────────────────────

def get_player_id(first_name, last_name):
    try:
        results = playerid_lookup(last_name, first_name, fuzzy=True)
        if results.empty:
            return None
        results = results.sort_values("mlb_played_last", ascending=False)
        return int(results.iloc[0]["key_mlbam"])
    except Exception:
        return None

def get_xwoba(player_id, handedness, start_dt, end_dt):
    try:
        data = statcast_batter(start_dt, end_dt, player_id)
    except Exception:
        return None
    if data is None or data.empty:
        return None
    vs_hand = data[data["p_throws"] == handedness]
    if vs_hand.empty:
        return None
    col = "estimated_woba_using_speedangle"
    values = vs_hand[col].dropna()
    if values.empty:
        return None
    return round(float(values.mean()), 3)

def xwoba_grade(val):
    if val is None:   return "—"
    if val >= 0.400:  return "🔥 Elite"
    if val >= 0.360:  return "✅ Great"
    if val >= 0.320:  return "🟢 Above Avg"
    if val >= 0.290:  return "🟡 Average"
    if val >= 0.250:  return "🟠 Below Avg"
    return             "🔴 Poor"

# ── Run ────────────────────────────────────────────────────────────────────

start_dt = f"{YEAR}-03-01"
end_dt   = f"{YEAR}-11-30"
hand_label = "RHP" if HANDEDNESS == "R" else "LHP"

print(f"Fetching {YEAR} Statcast data vs {hand_label}...\n")

rows = []
for slot, full_name in enumerate(LINEUP, 1):
    parts = full_name.strip().split()
    first, last = parts[0], " ".join(parts[1:])
    print(f"  {slot}. {full_name}...", end=" ", flush=True)

    pid = get_player_id(first, last)
    if pid is None:
        print("❌ not found")
        rows.append({"#": slot, "Name": full_name, f"xwOBA vs {hand_label}": None, "Grade": "—"})
        continue

    xwoba = get_xwoba(pid, HANDEDNESS, start_dt, end_dt)
    if xwoba is None:
        print("⚠️  no data")
    else:
        print(f"✅ {xwoba:.3f}")

    rows.append({
        "#": slot,
        "Name": full_name,
        f"xwOBA vs {hand_label}": f"{xwoba:.3f}" if xwoba else "N/A",
        "Grade": xwoba_grade(xwoba)
    })

# ── Display table ─────────────────────────────────────────────────────────

df = pd.DataFrame(rows)
print()
display(df.style.set_properties(**{
    "text-align": "left",
    "font-size": "14px",
    "padding": "6px 16px"
}).set_table_styles([
    {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "14px"), ("padding", "6px 16px")]}
]).hide(axis="index"))

# ── Summary ───────────────────────────────────────────────────────────────

xwoba_vals = [float(r[f"xwOBA vs {hand_label}"]) for r in rows if r[f"xwOBA vs {hand_label}"] not in (None, "N/A")]
if xwoba_vals:
    avg = sum(xwoba_vals) / len(xwoba_vals)
    print(f"\nLineup avg xwOBA vs {hand_label}: {avg:.3f}  {xwoba_grade(avg)}")
    print(f"League avg xwOBA (reference):   0.315")
